# Modul 7: Discrete Kalman Filter & Representasi State-Space
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 7 (Minggu 7):** Memformulasikan model *State-Space* linier diskrit dan mengimplementasikan *Discrete Kalman Filter* (*Time Update* & *Measurement Update*) dari nol untuk pemulihan sinyal sensor & estimasi *State-of-Charge* (SoC) baterai BESS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['axes.grid'] = True

class DiscreteKalmanFilter:
    def __init__(self, A, B, H, Q, R, x0, P0):
        self.A, self.B, self.H = A, B, H
        self.Q, self.R = Q, R
        self.x, self.P = x0.copy(), P0.copy()

    def predict(self, u):
        # Time Update (Prediksi)
        self.x = self.A @ self.x + self.B @ u
        self.P = self.A @ self.P @ self.A.T + self.Q
        return self.x.copy()

    def update(self, z):
        # Measurement Update (Koreksi)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        innovation = z - self.H @ self.x
        self.x = self.x + K @ innovation
        I = np.eye(self.P.shape[0])
        self.P = (I - K @ self.H) @ self.P
        return self.x.copy(), K

## 1. Simulasi Estimasi State-of-Charge (SoC) Baterai BESS dengan Sensor Bising

In [ ]:
rng = np.random.default_rng(77)
steps = 120
dt = 1.0 / 60.0  # 1 menit dalam satuan jam
C_nom = 100.0    # Kapasitas nominal 100 Ah

# Arus pengosongan bervariasi (A)
I_load = 25.0 + 10.0 * np.sin(np.linspace(0, 4 * np.pi, steps))

# Ground truth SoC + pengukuran sensor tegangan/SoC bising
soc_true = np.zeros(steps)
soc_true[0] = 0.90
for k in range(1, steps):
    soc_true[k] = soc_true[k - 1] - (dt / C_nom) * I_load[k - 1] + rng.normal(0, 0.001)

soc_meas = soc_true + rng.normal(0, 0.035, steps)

# Model State-Space: x_k = A x_{k-1} + B u_{k-1}, z_k = H x_k
A = np.array([[1.0]])
B = np.array([[-dt / C_nom]])
H = np.array([[1.0]])
Q = np.array([[1e-5]])      # Kovariansi derau proses
R = np.array([[0.035**2]])  # Kovariansi derau sensor

kf = DiscreteKalmanFilter(A, B, H, Q, R, x0=np.array([[0.85]]), P0=np.array([[0.05]]))

soc_est, gains = [], []
for k in range(steps):
    kf.predict(np.array([[I_load[k]]]))
    x_hat, K = kf.update(np.array([[soc_meas[k]]]))
    soc_est.append(x_hat[0, 0])
    gains.append(K[0, 0])

soc_est = np.array(soc_est)
rmse_raw = np.sqrt(np.mean((soc_true - soc_meas) ** 2)) * 100
rmse_kf = np.sqrt(np.mean((soc_true - soc_est) ** 2)) * 100
print(f"RMSE Sensor Mentah : {rmse_raw:.2f}% SoC")
print(f"RMSE Kalman Filter : {rmse_kf:.2f}% SoC (Reduksi Galat {(1 - rmse_kf/rmse_raw)*100:.1f}%)")

fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
axes[0].plot(soc_true * 100, 'k-', lw=2, label='SoC Aktual (Ground Truth)')
axes[0].scatter(np.arange(steps), soc_meas * 100, s=15, alpha=0.5, color='crimson', label='Sensor Bising')
axes[0].plot(soc_est * 100, color='emerald' if False else '#10b981', lw=2.2, label='Estimasi Kalman Filter')
axes[0].set_ylabel('SoC (%)')
axes[0].set_title('Pemulihan Sinyal State-of-Charge Baterai BESS dengan Discrete Kalman Filter')
axes[0].legend()

axes[1].plot(gains, color='navy', lw=2)
axes[1].set_ylabel('Kalman Gain $K_k$')
axes[1].set_xlabel('Langkah Waktu (Menit)')
axes[1].set_title('Konvergensi Kalman Gain')
plt.tight_layout()
plt.show()

## 2. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Ubah rasio $Q/R$ (misal perbesar $Q$ 100 kali atau perbesar $R$ 100 kali). Jelaskan efeknya pada *Kalman Gain* $K_k$ dan kelambatan (*lag*) respons estimasi!
2. **[C6]** Kembangkan model menjadi 2-state $\mathbf{x}_k = [\text{Posisi},\; \text{Kecepatan}]^T$ sesuai kode di Buku Praktikum Bab 7 dan ukur reduksi RMSE-nya!